# 04 — Anomaly Detection

> **Objective:** Detect extreme weather patterns using Z-score (univariate) and Isolation Forest (multivariate). Compare methods, visualize temporal and geographic distributions.

---

## 1. Setup

Import libraries and load the raw CSV (keeps physical units and coordinates).

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

project_root = Path.cwd().parent
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from weather_forecast.data_loader import load_raw_weather

reports_dir = project_root / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

# Load and validate data
df = load_raw_weather(project_root)

required = ["temperature_celsius", "latitude", "longitude", "last_updated"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df.dropna(subset=["last_updated", "temperature_celsius"]).copy()

print("Shape after dropna:", df.shape)
print("Time range:", df["last_updated"].min(), "->", df["last_updated"].max())
print("Lat/Lon non-null:", df["latitude"].notna().sum(), df["longitude"].notna().sum())
display(df[required + ["humidity", "wind_kph", "precip_mm"]].head(3) if "humidity" in df.columns else df[required].head(3))

## 2. Z-Score Detection (Temperature)

Flag rows where |z| exceeds threshold (global mean/std). Simple but assumes single distribution.

In [ ]:
Z_THRESHOLD = 3.0
temp = df["temperature_celsius"].astype(float)
mu, sigma = float(temp.mean()), float(temp.std(ddof=0))
sigma = max(sigma, 1e-9)
df["z_temp"] = (temp - mu) / sigma
df["anomaly_zscore"] = df["z_temp"].abs() > Z_THRESHOLD

n_z = int(df["anomaly_zscore"].sum())
print(f"Global temp mean={mu:.2f}°C, std={sigma:.2f}°C")
print(f"Z-score anomalies (|z|>{Z_THRESHOLD}): {n_z} ({100*n_z/len(df):.2f}%)")

## 3. Isolation Forest Detection (Multivariate)

Fit Isolation Forest on standardized numeric features. Captures joint anomalies, not just temperature extremes.

In [ ]:
feature_cols = [
    "temperature_celsius",
    "humidity",
    "wind_kph",
    "pressure_mb",
    "precip_mm",
]
feature_cols = [c for c in feature_cols if c in df.columns]
if len(feature_cols) < 3:
    raise ValueError(f"Need at least 3 features; got {feature_cols}")

X = df[feature_cols].copy()
X = X.fillna(X.median())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

CONTAMINATION = 0.02
iso = IsolationForest(
    n_estimators=200,
    contamination=CONTAMINATION,
    random_state=42,
    n_jobs=-1,
)
pred = iso.fit_predict(X_scaled)
df["anomaly_if"] = pred == -1
df["if_score"] = iso.score_samples(X_scaled)

n_if = int(df["anomaly_if"].sum())
print("Features used:", feature_cols)
print(f"Isolation Forest anomalies: {n_if} ({100*n_if/len(df):.2f}%)")

## 4. Method Comparison

Overlap analysis: both methods agree, only Z-score, only IF, or neither.

In [ ]:
both = df["anomaly_zscore"] & df["anomaly_if"]
only_z = df["anomaly_zscore"] & ~df["anomaly_if"]
only_if = ~df["anomaly_zscore"] & df["anomaly_if"]
neither = ~(df["anomaly_zscore"] | df["anomaly_if"])

cmp = pd.DataFrame(
    {
        "count": [both.sum(), only_z.sum(), only_if.sum(), neither.sum()],
        "label": ["both", "zscore_only", "if_only", "neither"],
    }
).set_index("label")
display(cmp.T)

summary_md = f"""
| Segment | Count | Share |
|---|---:|---:|
| Both | {both.sum()} | {100*both.mean():.2f}% |
| Z-score only | {only_z.sum()} | {100*only_z.mean():.2f}% |
| IF only | {only_if.sum()} | {100*only_if.mean():.2f}% |
| Neither | {neither.sum()} | {100*neither.mean():.2f}% |
"""
print(summary_md)

## 5. Temporal Visualization

Temperature time series with anomalies highlighted (subsampled for plotting).

In [ ]:
MAX_POINTS = 25_000
plot_df = df.sample(n=min(MAX_POINTS, len(df)), random_state=42).sort_values("last_updated")

normal_mask = ~(plot_df["anomaly_zscore"] | plot_df["anomaly_if"])
base = plot_df.loc[normal_mask]
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=base["last_updated"],
        y=base["temperature_celsius"],
        mode="markers",
        marker=dict(size=4, color="lightgray"),
        name="Normal (sample)",
    )
)
for name, mask, color in [
    ("Z-score", plot_df["anomaly_zscore"], "crimson"),
    ("Isolation Forest", plot_df["anomaly_if"], "darkorange"),
]:
    sub = plot_df[mask]
    if len(sub) == 0:
        continue
    fig.add_trace(
        go.Scatter(
            x=sub["last_updated"],
            y=sub["temperature_celsius"],
            mode="markers",
            marker=dict(size=7, color=color),
            name=f"Anomaly ({name})",
        )
    )

fig.update_layout(
    title="Temperature vs time — anomalies (subsample for plotting)",
    xaxis_title="last_updated",
    yaxis_title="Temperature (°C)",
    height=520,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
out_ts = reports_dir / "anomaly_temperature_timeseries.html"
fig.write_html(out_ts)
print("Saved:", out_ts)
try:
    fig.show()
except ValueError as exc:
    if "nbformat" in str(exc).lower():
        print("Open HTML in browser for interactive plot.")
    else:
        raise

## 6. Geographic Visualization

World map scatter plot showing anomaly distribution by location.

In [ ]:
geo_df = df.dropna(subset=["latitude", "longitude"]).copy()
if len(geo_df) > 40_000:
    geo_df = geo_df.sample(40_000, random_state=42)

def anomaly_label(row):
    if row["anomaly_zscore"] and row["anomaly_if"]:
        return "both"
    if row["anomaly_zscore"]:
        return "zscore_only"
    if row["anomaly_if"]:
        return "if_only"
    return "normal"

geo_df["anomaly_group"] = geo_df.apply(anomaly_label, axis=1)

fig_map = px.scatter_geo(
    geo_df,
    lat="latitude",
    lon="longitude",
    color="anomaly_group",
    color_discrete_map={
        "normal": "lightgray",
        "both": "black",
        "zscore_only": "crimson",
        "if_only": "darkorange",
    },
    title="Geographic distribution of anomalies (subsample)",
    height=600,
)
fig_map.update_geos(showland=True, landcolor="whitesmoke")
out_map = reports_dir / "anomaly_geographic_distribution.html"
fig_map.write_html(out_map)
print("Saved:", out_map)
try:
    fig_map.show()
except ValueError as exc:
    if "nbformat" in str(exc).lower():
        print("Open HTML in browser for interactive map.")
    else:
        raise

## 7. Static Comparison Chart

Bar chart of method overlap counts for documentation.

In [ ]:
counts = {
    "both": int(both.sum()),
    "zscore_only": int(only_z.sum()),
    "if_only": int(only_if.sum()),
    "neither": int(neither.sum()),
}
plt.figure(figsize=(8, 4))
sns.barplot(x=list(counts.keys()), y=list(counts.values()))
plt.title("Anomaly detection — method overlap (counts)")
plt.ylabel("Count")
plt.xlabel("Segment")
plt.tight_layout()
out_bar = reports_dir / "anomaly_method_comparison.png"
plt.savefig(out_bar, dpi=150)
plt.show()
print("Saved:", out_bar)

## 8. Summary

**Key findings:**
- **Z-score:** Flags univariate extremes vs global mean; mixing climate zones inflates false positives.
- **Isolation Forest:** Captures multivariate outliers; not limited to temperature alone.
- **Tuning:** Adjust `Z_THRESHOLD`, `CONTAMINATION`, or use per-region Z-scores for fairer comparison.

In [ ]:
exported = [
    reports_dir / "anomaly_temperature_timeseries.html",
    reports_dir / "anomaly_geographic_distribution.html",
    reports_dir / "anomaly_method_comparison.png",
]
for f in exported:
    print(f"{f.name}: {'OK' if f.exists() else 'MISSING'}")

In [ ]:
# Export real anomalies section (issue #0015).
from datetime import datetime, timezone

from weather_forecast.dashboard_export import build_anomalies_real, write_real_section

_web_data = project_root / "web" / "public" / "data"
_gen_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

_n = len(df)
_zscore = {"threshold": float(Z_THRESHOLD), "count": int(n_z), "share_pct": round(100 * n_z / _n, 2)}
_iso = {"contamination": float(CONTAMINATION), "count": int(n_if), "share_pct": round(100 * n_if / _n, 2)}
_overlap = int((df["anomaly_zscore"] & df["anomaly_if"]).sum())

_country_col = "country" if "country" in df.columns else None
_anom = df[df["anomaly_zscore"] | df["anomaly_if"]].dropna(subset=["latitude", "longitude"])
_anom = _anom.sample(min(150, len(_anom)), random_state=42)


def _detected(r):
    if r["anomaly_zscore"] and r["anomaly_if"]:
        return "both"
    return "zscore" if r["anomaly_zscore"] else "isolation_forest"


_records = []
for _, _r in _anom.iterrows():
    _ts = _r["last_updated"]
    _records.append(
        {
            "ts": _ts.strftime("%Y-%m-%dT%H:%M:%SZ") if pd.notna(_ts) else "1970-01-01T00:00:00Z",
            "country": str(_r[_country_col]) if _country_col else "Unknown",
            "lat": round(float(_r["latitude"]), 4),
            "lon": round(float(_r["longitude"]), 4),
            "temp_c": round(float(_r["temperature_celsius"]), 2),
            "z": round(float(_r["z_temp"]), 3),
            "if_score": round(float(_r["if_score"]), 4),
            "detected_by": _detected(_r),
        }
    )
_anom_real = build_anomalies_real(
    zscore=_zscore,
    isolation_forest=_iso,
    overlap_count=_overlap,
    records=_records,
    generated_at=_gen_at,
)
write_real_section(_web_data, "anomalies", _anom_real)
print("Wrote real anomalies:", _zscore["count"], _iso["count"], _overlap)
